# TruFor + ViT clasificador — Colab T4

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.

Pipeline:

1. imagen → **preprocesamiento** (`src/preprocess.py`, variantes antes de TruFor)
2. **TruFor** (congelado) → score global + mapa de manipulación + mapa de confianza
3. **normalización de tamaño** de los mapas a `MAP_SIZE`×`MAP_SIZE` (`src/maps.py`) y cache
4. **ViT desde cero** (RGB + mapa + confianza) → probabilidad de manipulación

Lo que hay que conservar entre sesiones vive en Drive (`BASE`); el disco de Colab
se borra. Las celdas de **extracción** y **entrenamiento** son *reanudables*: si la
sesión se corta, volvé a correr 1→4, la de empaquetado y la que se interrumpió.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Cambiá el entorno de ejecución a GPU (T4)"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/Valentin190306/Deepfake_Detection_PDI_2026.git"
BASE = "/content/drive/MyDrive/pdi"          # datos, cache y checkpoints (persistente)

VARIANT = "raw"                               # variantes en src/preprocess.py
MAP_SIZE = 384                                # lado S de entrada del ViT
MAX_SIDE = 1280                               # límite de memoria para TruFor en T4
CACHE_KEY = f"{VARIANT}__s{MAP_SIZE}__ms{MAX_SIDE}"
PACKED = f"/content/packed/{VARIANT}__s{MAP_SIZE}"   # local: se regenera en ~2 min
print(CACHE_KEY, "->", PACKED)

## 1. Repo, dependencias y modelo TruFor

No se reinstala `torch`: Colab ya trae una build con CUDA y reinstalarlo suele
romper la GPU. `requirements.txt` fija `timm==0.9.16` porque TruFor usa la API
vieja de timm.

In [ ]:
import os
os.makedirs(BASE, exist_ok=True)
!test -d /content/repo || git clone -q {REPO_URL} /content/repo
%cd /content/repo
!git pull -q --ff-only || echo "(pull no aplicable)"
!git log --oneline -1
!python --version
!pip install -q -r requirements.txt

In [ ]:
!bash scripts/get_trufor.sh

## 2. Verificación del entorno

Tiene que dar ~`0.19` para las *pristine* y ~`0.99` para las *tampered* (en GPU los
decimales varían un poco). Si no coincide, no sigas: algo del entorno está mal.

In [ ]:
!python src/detect.py third_party/trufor/images/

## 3. Datos: IMD2020 desde Kaggle

Dataset `saikotahmed/imd2020`: 2010 manipuladas con máscara + 414 auténticas.
Si la descarga anónima falla, creá en **Secretos** de Colab (ícono de llave)
`KAGGLE_USERNAME` y `KAGGLE_KEY` y habilitá el acceso al notebook.

La preparación copia las imágenes a `{BASE}/data` **una sola vez** y arma el
`manifest.csv` con etiqueta y **grupo de escena**. El grupo es lo que evita que
un original quede en train y su versión manipulada en validación.

In [ ]:
import os, kagglehub
try:
    imd_path = kagglehub.dataset_download("saikotahmed/imd2020")
except Exception as e:
    print("Descarga anónima falló:", e)
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    imd_path = kagglehub.dataset_download("saikotahmed/imd2020")
print(imd_path)

In [ ]:
!python src/prepare_imd2020.py --src {imd_path} --out {BASE}/data --prefix imd_

In [ ]:
import pandas as pd
m = pd.read_csv(f"{BASE}/data/manifest.csv")
print(m.groupby("source").size().to_string())
print("grupos:", m.group.nunique())
display(m.head())

## 4. Extracción de mapas TruFor (reanudable)

Una pasada por imagen: variante → TruFor → normalización a `MAP_SIZE` → `.npz` en
Drive. Se saltea lo ya procesado, así que si se corta se vuelve a lanzar.

En T4, con `MAX_SIDE=1280`, tarda ~30–60 min para las 2424 imágenes. `MAX_SIDE`
evita OOM con las fotos de ~3000×4400 px; TruFor trabaja a resolución nativa y el
ViT de todos modos recibe 384×384.

In [ ]:
!python src/extract_maps.py --manifest {BASE}/data/manifest.csv --cache {BASE}/cache --variants {VARIANT} --map-size {MAP_SIZE} --max-side {MAX_SIDE}

## 5. Empaquetado local del cache

Leer 2400 `.npz` desde Drive en cada época es lento y frágil. Este paso los junta
en un único `.npy` memmap en el disco local de Colab (~1.8 GB, ~2 min). **Hay que
repetirlo al inicio de cada sesión** (o después de agregar imágenes).

In [ ]:
!python src/pack_cache.py --cache {BASE}/cache/{CACHE_KEY} --out {PACKED} --overwrite

## 6. Prueba rápida (3 épocas, 20 pasos)

Sirve para confirmar que baja la loss y que el split tiene las dos clases. No
mires las métricas: son de juguete.

In [ ]:
!python src/train.py --manifest {BASE}/data/manifest.csv --packed {PACKED} --splits {BASE}/runs/splits.json --out {BASE}/runs/smoke --size {MAP_SIZE} --epochs 3 --limit-steps 20 --patience 0

## 7. Entrenamiento completo (reanudable)

Checkpoints en `{BASE}/runs/vit/`: `last.pt` (cada época), `best.pt` (mejor AUC de
validación), `config.json` y `log.csv`. Si Colab se desconecta, volvé a correr esta
celda: retoma desde `last.pt`.

El ViT es **desde cero** y arranca igual al baseline de TruFor (logit residual con
`alpha=1`); a partir de ahí aprende una corrección. `--patience 20` corta si deja
de mejorar.

In [ ]:
!python src/train.py --manifest {BASE}/data/manifest.csv --packed {PACKED} --splits {BASE}/runs/splits.json --out {BASE}/runs/vit --size {MAP_SIZE} --epochs 150 --bs 16 --workers 2

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv(f"{BASE}/runs/vit/log.csv")
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for c in ["acc", "bal_acc", "f1"]:
    ax[0].plot(log.epoch, log[c], label=c)
ax[0].set_title("clasificación"); ax[0].legend(); ax[0].grid(alpha=.3)
for c in ["auc", "ap"]:
    ax[1].plot(log.epoch, log[c], label=c)
ax[1].set_title("ranking"); ax[1].legend(); ax[1].grid(alpha=.3)
ax[2].plot(log.epoch, log.train_loss, label="train")
ax[2].plot(log.epoch, log.val_loss, label="val")
ax[2].set_title("loss"); ax[2].legend(); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 8. Evaluación

Métricas sobre la validación de IMD2020, matriz de confusión, ROC/PR y una
galería con los errores más confiados.

> ⚠️ **TruFor se entrenó con IMD2020**, así que estas métricas están infladas.
> El número honesto es el de un test externo (ver la última sección).

In [ ]:
!python src/eval.py --manifest {BASE}/data/manifest.csv --packed {PACKED} --ckpt {BASE}/runs/vit/best.pt --splits {BASE}/runs/splits.json --out {BASE}/runs/eval --split val --gallery 6

In [ ]:
from IPython.display import Image as Img, display
display(Img(filename=f"{BASE}/runs/eval/metricas.png"))
display(Img(filename=f"{BASE}/runs/eval/galeria.png"))

## 9. Test externo (recomendado)

Para reportar un número honesto hace falta un conjunto que **no** sea IMD2020
(por ejemplo CASIA, Columbia, CoMoFoD o fotos propias manipuladas). El flujo es
el mismo: armar un `manifest.csv` con las columnas `file, mask, label, group,
source, width, height`, correr `extract_maps.py` + `pack_cache.py` sobre él y
evaluar con `--split all`.

```bash
python src/extract_maps.py --manifest /ruta/externo/manifest.csv     --cache {BASE}/cache_ext --variants raw --map-size 384 --max-side 1280
python src/pack_cache.py --cache {BASE}/cache_ext/raw__s384__ms1280 --out /content/packed/ext
python src/eval.py --manifest /ruta/externo/manifest.csv --packed /content/packed/ext     --ckpt {BASE}/runs/vit/best.pt --out {BASE}/runs/eval_ext --split all
```